# CDK20に類似した、データが豊富な蛋白質の探索


## Why: CDK20 itself has almost no data

CDK20 (`Q8IZL9`) has zero RCSB PDB structures and, per UniProt's own PDB
cross-reference list, zero there too. ChEMBL likewise has essentially no
bioactivity data for it (checked below: 1 record with a `pchembl_value`).
Before any pocket/docking work on CDK20 itself can be validated against real
structures or SAR, it's worth finding a close homolog that actually has
both -- a structural surrogate for docking, and/or an activity-rich target
to borrow chemical matter/SAR intuition from.

This notebook: (1) pulls CDK20's own UniProt family classification, (2)
BLASTs its sequence against PDB and Swiss-Prot via EBI's Job Dispatcher to
find homologs, (3) narrows to human paralogs, and (4) cross-checks each
candidate's actual RCSB structure count and ChEMBL bioactivity count -- so
the final choice is driven by data availability, not just sequence
identity.

## CDK20's own UniProt annotation

In [ ]:
import requests

r = requests.get("https://rest.uniprot.org/uniprotkb/Q8IZL9.json", timeout=30)
entry = r.json()

full_name = entry["proteinDescription"]["recommendedName"]["fullName"]["value"]
similarity = next(
    (t["value"] for c in entry["comments"] if c["commentType"] == "SIMILARITY" for t in c["texts"]),
    None,
)
n_pdb_xrefs = sum(1 for x in entry["uniProtKBCrossReferences"] if x["database"] == "PDB")
seq_length = entry["sequence"]["length"]

print("Protein:", full_name)
print("Family:", similarity)
print("Sequence length:", seq_length)
print("PDB cross-references in UniProt:", n_pdb_xrefs)

## BLASTP search via chem.blast.blastp

`chem.blast.blastp` wraps [EBI's Job Dispatcher REST API](https://www.ebi.ac.uk/jdispatcher/docs/webservices/)
(submit, poll, fetch/parse results in one call): `blastp` against `pdb`
(finds homologs that already have a solved structure -- direct
docking-template candidates) and against `uniprotkb_swissprot` (broadens to
every reviewed/well-annotated homolog regardless of whether it has a
structure -- candidate sources of bioactivity/SAR data). The API requires a
contact `email`.


In [ ]:
EBI_EMAIL = "hara.ryuichiro@gmail.com"  # required by the EBI Job Dispatcher API

fasta = requests.get("https://rest.uniprot.org/uniprotkb/Q8IZL9.fasta", timeout=30).text
print(fasta)


In [ ]:
from chem import blast

pdb_hits = blast.blastp(fasta, database="pdb", email=EBI_EMAIL, title="cdk20_vs_pdb")
sp_hits = blast.blastp(fasta, database="uniprotkb_swissprot", email=EBI_EMAIL, title="cdk20_vs_swissprot")


### Top hits against PDB

Every hit here already has a solved structure -- a direct pool of candidate
docking templates, ranked by sequence identity to CDK20.

In [ ]:
import pandas as pd

pdb_hits_df = pd.DataFrame(pdb_hits)
sp_hits_df = pd.DataFrame(sp_hits)

print(f"{len(pdb_hits_df)} PDB hits, {len(sp_hits_df)} Swiss-Prot hits (e-value <= 1e-10, top 50 each)")
display(pdb_hits_df.head(15).style.hide(axis="index"))

### Human paralogs among the Swiss-Prot hits

The PDB-hit list above is dominated by non-human/non-paralog kinases (yeast
Cdc2, malaria parasite CDK2 homologs, plant CDKs, ...) that happen to share
the same catalytic domain fold -- not useful as a source of human SAR data.
Filtering the broader Swiss-Prot hit list down to `_HUMAN` entries (and
dropping the CDK20 self-hit) narrows this to CDK20's actual human
paralogs.

In [ ]:
human_paralogs_df = sp_hits_df[
    sp_hits_df["description"].str.contains("_HUMAN") & (sp_hits_df["accession"] != "Q8IZL9")
].reset_index(drop=True)
display(human_paralogs_df.style.hide(axis="index"))

## Ranking candidates by actual data availability

Sequence identity alone doesn't say whether a homolog is *useful* -- CDK3 is
the closest human paralog by identity but, as seen below, is barely better
characterized than CDK20 itself. Each human paralog's UniProt accession is
cross-checked against the RCSB Search API (`chem.rcsb`'s own
`_search_entry_ids` helper -- structure count only, no download) and the
ChEMBL activity endpoint (count of records with a `pchembl_value`, i.e.
usable potency data) to see which candidate is actually backed by data.

In [ ]:
from chem.ids import resolve_target_chembl_id
from chem.rcsb.fetch import _search_entry_ids

CHEMBL_API = "https://www.ebi.ac.uk/chembl/api/data"


def chembl_activity_count(chembl_target_id):
    r = requests.get(
        f"{CHEMBL_API}/activity.json",
        params={
            "target_chembl_id": chembl_target_id,
            "pchembl_value__isnull": "false",
            "format": "json",
            "limit": 1,
        },
        timeout=30,
    )
    return r.json()["page_meta"]["total_count"]


candidates = [("Q8IZL9", "CDK20_HUMAN (query)", None)] + [
    (row.accession, row.description.split()[1], row.identity_pct) for row in human_paralogs_df.itertuples()
]

rows = []
for acc, label, identity_pct in candidates:
    n_pdb = len(_search_entry_ids(acc))
    chembl_id = resolve_target_chembl_id(acc)
    n_activities = chembl_activity_count(chembl_id)
    rows.append(
        {
            "target": label,
            "uniprot": acc,
            "identity_pct": identity_pct,
            "chembl_target_id": chembl_id,
            "n_pdb_structures": n_pdb,
            "n_chembl_activities": n_activities,
        }
    )

richness_df = pd.DataFrame(rows).sort_values("n_chembl_activities", ascending=False).reset_index(drop=True)
display(richness_df.style.hide(axis="index"))

## Conclusion

**`CDK2_HUMAN` (`P24941`) is the practical surrogate**: 43.8% identity to
CDK20 (close to the best any human paralog reaches), **522** RCSB
structures, and **3454** ChEMBL activity records with a `pchembl_value` --
overwhelmingly the richest of any candidate here, structurally and in SAR
terms. It's a standard, extensively characterized kinase, so both a
crystal-structure-based active site (reusing the ligand-position-consensus
approach from the thrombin notebook, `alphafold_pocket_thrb_human.ipynb`)
and a large training set for activity modeling are readily available.

Two runner-up notes:
- `CDK3_HUMAN` has the *highest* raw identity (45.4%) but only 2 PDB
  structures and 59 activities -- barely more data than CDK20 itself, so
  identity alone would have picked a dead end here.
- `CDK7_HUMAN` (42.8% identity, 56 structures, 824 activities) is the
  closest *functional* analog -- UniProt's FUNCTION comment notes CDK20
  activates CDK2 by phosphorylating its T-loop (Thr-160), the same
  CDK-activating-kinase (CAK) role CDK7 plays canonically -- worth
  considering if mechanistic similarity matters more than raw sequence
  identity for a given downstream use.